In [576]:
import os
import pandas as pd
import json
from datetime import datetime
import matplotlib.pyplot as plt
from tqdm import tqdm_notebook

In [577]:
def make_directory(directory, verbose=True):
    """
    Create a directory
    :param directory: path to the folder to create
    :type directory: str
    :param verbose: verbose
    :type verbose: bool
    :return: None
    :rtype: object
    """
    if not (os.path.exists(directory)):
        os.makedirs(directory)
        if verbose:
            print(f"Created a directory: '{directory}'")

In [578]:
client = "cwallet"
DATA_DIR = f"data/{client}"
EVALUATION_FOLDER = f"evaluation/{client}"
make_charts = False
make_directory(EVALUATION_FOLDER, verbose=False)

In [579]:
def ts_converter(x):
    return datetime.fromtimestamp(int(float(x) / 1000))

def compose_topic_name(source, source_type="log"):
    items = source.split("-")
    infra = items[0]
    agent = items[1]
    component = "-".join(items[2:])
    return f"{infra}-{agent}-{source_type}-{component}-stashed"

def parse_source(source):
    items = source.split("-")
    infra = items[0]
    agent = items[1]
    component = "-".join(items[2:])
    return infra, agent, component

## DBSCAN anomalies

In [580]:
dbscan_file_path = os.path.join(DATA_DIR, "dbscan.csv")
dbscan_anomalies = pd.read_csv(dbscan_file_path,parse_dates=["datetime"],)

In [581]:
(
    dbscan_anomalies["infra"],
    dbscan_anomalies["agent"],
    dbscan_anomalies["component"],
) = zip(*dbscan_anomalies["source"].apply(lambda x: parse_source(x)))
dbscan_anomalies["datetime"] = pd.to_datetime(dbscan_anomalies["timestamp"], unit="s")

In [582]:
dbscan_anomalies.head()

,timestamp,n_message,isAnomaly,source,datetime,value,infra,agent,component
0,1636553700,1117,True,60ed840f588aa700127280d6-linux60effb26a06f3200...,2021-11-10 14:15:00,1117,60ed840f588aa700127280d6,linux60effb26a06f320011336760,nginx22d27dc8e4237c4f
1,1636553760,1113,True,60ed840f588aa700127280d6-linux60effb26a06f3200...,2021-11-10 14:16:00,1113,60ed840f588aa700127280d6,linux60effb26a06f320011336760,nginx22d27dc8e4237c4f
2,1636554600,1498,True,60ed840f588aa700127280d6-linux60effb26a06f3200...,2021-11-10 14:30:00,1498,60ed840f588aa700127280d6,linux60effb26a06f320011336760,nginx22d27dc8e4237c4f
3,1636554660,1495,True,60ed840f588aa700127280d6-linux60effb26a06f3200...,2021-11-10 14:31:00,1495,60ed840f588aa700127280d6,linux60effb26a06f320011336760,nginx22d27dc8e4237c4f
4,1636557120,2333,True,60ed840f588aa700127280d6-linux60effb26a06f3200...,2021-11-10 15:12:00,2333,60ed840f588aa700127280d6,linux60effb26a06f320011336760,nginx22d27dc8e4237c4f


## Anomalies registered in the IM

In [583]:
def clean_log_line(ll):
    ll = ll.replace("Anomaly: ", "").replace("\n", "")
    ll = json.loads(ll)
    return ll

In [584]:
client_log_file_path = os.path.join(DATA_DIR, "IM.log")
with open(client_log_file_path, "r") as f:
    lines = f.readlines()

clean_anomaly_logs = list(
    map(clean_log_line, filter(lambda x: "Anomaly: " in x, lines))
)



In [585]:
minima_anomalies = list(
    filter(lambda x: "log frequency" in x.get("type", ""), clean_anomaly_logs)
)

In [586]:
start_ts = dbscan_anomalies["timestamp"].min()
end_ts = dbscan_anomalies["timestamp"].max()

In [587]:
formatted_docs = []
for doc in minima_anomalies:
    current_ts = int(doc["timestamp"] / 1000)
    if current_ts < start_ts or current_ts > end_ts:
        continue
    d = {
        "timestamp": current_ts,
        "source": dbscan_anomalies[
            (dbscan_anomalies["infra"] == doc["infrastructureId"])
            & (dbscan_anomalies["component"] == doc["ITComponentId"])
        ]["source"].values.any(),
        "infra": doc["infrastructureId"],
        "component": doc["ITComponentId"],
        "value": doc["value"],
    }
    formatted_docs.append(d)
if formatted_docs:
    minima_anomalies_df = pd.DataFrame(formatted_docs)
    minima_anomalies_df["datetime"] = pd.to_datetime(
        minima_anomalies_df["timestamp"], unit="s"
    )
else:
    minima_anomalies_df = pd.DataFrame(
        [], columns=["timestamp", "source", "infra", "component", "value", "datetime",],
    )

In [588]:
minima_anomalies_df.head()

,timestamp,source,infra,component,value,datetime


## Deployments

In [589]:
lad_logfrequency_models_path = os.path.join(DATA_DIR, "lad_logfrequency_models.json")
lad_logfrequency_models = pd.read_json(lad_logfrequency_models_path)

In [590]:
lad_logfrequency_models["deployed models"] = lad_logfrequency_models["models"].apply(
    lambda x: ", ".join(x.keys())
)
lad_logfrequency_models["source"] = lad_logfrequency_models["topic"].apply(
    lambda x: x.replace("-log-", "-").replace("-stashed", "")
)

In [591]:
deployed_logfrequency_models = lad_logfrequency_models[["source", "shape", "deployed models"]].set_index("source").to_dict(orient="index")

## Log frequency data

In [592]:
logfrequency_data_dir = os.path.join(DATA_DIR, "logfrequency_data")

In [593]:
train_date_end = "2021-11-08 14:50:00"

## Evaluation charts

In [594]:
for current_folder in ["anomalies", "no_anomalies", "not_deployed"]:
    make_directory(os.path.join(EVALUATION_FOLDER, current_folder))

if make_charts:
    for root, _, files in os.walk(logfrequency_data_dir):
        for file in tqdm_notebook(files):
            if file.endswith(".csv"):
                current_source = file.replace(".csv", "")
                filepath = os.path.join(root, file)
                df = pd.read_csv(
                    filepath,
                    parse_dates=["timestamp"],
                    date_parser=ts_converter,
                    usecols=["timestamp", "n_message"],
                )
                current_minima_anomalies_df = minima_anomalies_df[
                    minima_anomalies_df["source"] == current_source
                ]
                current_dbscan_anomalies_df = dbscan_anomalies[
                    (dbscan_anomalies["source"] == current_source)
                    & (
                        ~dbscan_anomalies["timestamp"].isin(
                            current_minima_anomalies_df["timestamp"]
                        )
                    )
                ]
                current_df = df.set_index("timestamp")
                train_df = current_df[current_df.index < train_date_end]
                test_df = current_df[current_df.index >= train_date_end]
                if train_df.shape[0] == 0 or test_df.shape[0] == 0:
                    print("Skip", current_source)
                    continue
                ax = train_df.plot(figsize=(15, 4), c="palegreen",)
                test_df.plot(c="royalblue", ax=ax)
                legend_names = ["train data", "test data"]
                if current_source not in deployed_logfrequency_models:
                    current_folder = "not_deployed"
                    current_title = f"{current_source}\nNo models deployed"
                else:
                    current_title = (
                        f"{current_source}\n{deployed_logfrequency_models[current_source]}"
                    )
                    if (
                        current_minima_anomalies_df.shape[0] > 0
                        or current_dbscan_anomalies_df.shape[0] > 0
                    ):
                        current_folder = "anomalies"
                        if current_minima_anomalies_df.shape[0] > 0:
                            current_minima_anomalies_df.plot.scatter(
                                x="datetime", y="value", c="red", ax=ax
                            )
                            legend_names.append("Anomaly shown on Minima")
                        if current_dbscan_anomalies_df.shape[0] > 0:
                            current_dbscan_anomalies_df.plot.scatter(
                                x="datetime", y="value", c="black", ax=ax
                            )
                            legend_names.append("Detected anomaly but not shown on Minima")
                    else:
                        current_folder = "no_anomalies"
                plt.legend(legend_names)
                plt.title(current_title)
                ax.get_figure().savefig(
                    os.path.join(
                        EVALUATION_FOLDER, current_folder, current_source + ".png"
                    ),
                    bbox_inches="tight",
                )
                plt.close()

/opt/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:6: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  


Skip 60ed840f588aa700127280d6-linux60effb30a06f320011336761-linux60effb30a06f320011336761
Skip 60ed840f588aa700127280d6-linux60effafba06f32001133675d-linux60effafba06f32001133675d
Skip 60ed840f588aa700127280d6-linux60effa84a06f32001133675a-linux60effa84a06f32001133675a
Skip 60ed840f588aa700127280d6-linux60effb26a06f320011336760-linux60effb26a06f320011336760
Skip 60ed840f588aa700127280d6-linux60effb4aa06f320011336762-rabbitmqeb234c270bf3d9cd
Skip 60ed840f588aa700127280d6-linux60effadaa06f32001133675c-linux60effadaa06f32001133675c
Skip 60ed840f588aa700127280d6-linux60effb4aa06f320011336762-linux60effb4aa06f320011336762



In [595]:
make_directory(os.path.join(EVALUATION_FOLDER, "anomaly_distribution"))

In [596]:
if make_charts:
    fig, axes = plt.subplots(2, figsize=(14,6), sharex=True)
    dbscan_anomalies.set_index("datetime").resample("1d").count()[["timestamp"]].plot(
        kind="bar", legend=False, title=f"Number of anomalies per day: {client}", ax=axes[0]
    )
    incidents.set_index("datetime").resample("1d").count()[["timestamp"]].plot(
        kind="bar", legend=False, title=f"Number of incidents per day: {client}", ax=axes[1]
    )
    plt.xticks(rotation=40)
    fig.savefig(
        os.path.join(
            EVALUATION_FOLDER, "anomaly_distribution", "merged_anomaliesPerDay.png"
        ),
        bbox_inches="tight",
    )
    plt.close()

In [597]:
if make_charts:
    ax = dbscan_anomalies.set_index("datetime").resample("1d").count()[["timestamp"]].plot(
        kind="bar", legend=False, title=f"Number of anomalies per day: {client}", figsize=(15, 4)
    )

    ax.get_figure().savefig(
        os.path.join(
            EVALUATION_FOLDER, "anomaly_distribution", "anomaliesPerDay.png"
        ),
        bbox_inches="tight",
    )
    plt.close()

In [598]:
if make_charts:
    ax=incidents.set_index("datetime").resample("1d").count()[["timestamp"]].plot(
        kind="bar", legend=False, title=f"Number of incidents per day: {client}", figsize=(15, 4)
    )
    ax.get_figure().savefig(
        os.path.join(
            EVALUATION_FOLDER, "anomaly_distribution", "incidentsPerDay.png"
        ),
        bbox_inches="tight",
    )
    plt.close()

## Report

### Incident count

In [599]:
dbscan_anomalies.datetime.min(), dbscan_anomalies.datetime.max()

(Timestamp('2021-11-08 09:13:00'), Timestamp('2021-11-11 07:41:00'))

In [600]:
def get_incidents(anomalies, wait_next_incident = 5*60):
    for col in {"datetime", "timestamp", "source"}:
        if col not in anomalies.columns:
            print(f"Column {col} was not found in provided DataFrame")
            return pd.DataFrame(), {}
    incidents_per_component = {}
    for source, df in anomalies.groupby("source"):
        _df = df.sort_values(by="timestamp")[["datetime", "timestamp"]].copy()
        _df["start_incident"] = (
            _df.diff()["timestamp"].fillna(2 * wait_next_incident) > wait_next_incident
        )
        incidents_per_component[source] = _df[_df["start_incident"]].shape[0]

    _df = anomalies.sort_values(by="timestamp")[["datetime", "timestamp"]].copy()
    _df["start_incident"] = (
        _df.diff()["timestamp"].fillna(2 * wait_next_incident) > wait_next_incident
    )
    incidents = _df[_df["start_incident"]].reset_index(drop=True).copy()
    return incidents, incidents_per_component

In [601]:
incidents, incidents_per_component = get_incidents(
    dbscan_anomalies, wait_next_incident=5 * 60
)
minima_incidents, minima_incidents_per_component = get_incidents(
    minima_anomalies_df, wait_next_incident=5 * 60
)

In [605]:
print(client.capitalize(), "\n", "-" * 10, sep="")
print("Deployed models (covered topics):", lad_logfrequency_models.shape[0])
print("Overall number of detected anomalies:", dbscan_anomalies.shape[0])
print("Overall number of anomalies sent to the client:", minima_anomalies_df.shape[0])
print("\nIncidents:\n", "-" * 10, sep="")
print_d = {
    "overall": (incidents, incidents_per_component),
    "shown to the client": (minima_incidents, minima_incidents_per_component),
}
for label, data in print_d.items():
    print(label)
    print(f"\tNumber of incidents:", data[0].shape[0])
    print(f"\tAverage number of incidents per day:", round(data[0].shape[0] / 7, 2))
    print(
        f"\tAverage number of incidents per component per day:",
        round(sum(data[1].values()) / 7, 2),
    )

Cwallet
----------
Deployed models (covered topics): 21
Overall number of detected anomalies: 49
Overall number of anomalies sent to the client: 0

Incidents:
----------
overall
	Number of incidents: 10
	Average number of incidents per day: 1.43
	Average number of incidents per component per day: 2.71
shown to the client
	Number of incidents: 0
	Average number of incidents per day: 0.0
	Average number of incidents per component per day: 0.0


## Restore cwallet data

cwallet_p = "data/cwallet/temp/"
dfs = []
for root, _, files in os.walk(cwallet_p):
    for file in tqdm_notebook(files):
        if file.endswith("restore_merged.csv"):
            current_source = file.replace("-restore_merged.csv", "")
            filepath = os.path.join(root, file)
            df = pd.read_csv(
                filepath,
                parse_dates=["timestamp"],
#                 date_parser=ts_converter,
                usecols=["timestamp", "n_message", "isAnomaly"],
            )
            df["source"] = current_source
            dfs.append(df)

merged = pd.concat(dfs)
merged["datetime"] = pd.to_datetime(merged["timestamp"], unit="s")

merged[merged["isAnomaly"]].head()

merged["value"] = merged["n_message"]

merged[merged["isAnomaly"]].reset_index(drop=True).to_csv(
            os.path.join(cwallet_p, "dbscan.csv"), index=False
        )

df["source"].any()

lad_logfrequency_models = pd.read_pickle(
    "/Users/valentin.lapparov/PycharmProjects/ex-machina-meso/results/logfrequency/clustering/cwallet/2021-11-08/mongo_docs.pickle"
)

import json

with open("data/cwallet/lad_logfrequency_models.json", "w") as outfile:
    json.dump(lad_logfrequency_models, outfile)